In [ ]:
import os
import pandas as pd
from IPython.display import FileLink, display

backup_dir = "/kaggle/working/HAR_Research"
os.makedirs(backup_dir, exist_ok=True)

manifest_path = os.path.join(
    backup_dir,
    "ucf101_split01_manifest.csv"
)

df.to_csv(manifest_path, index=False)

print("Manifest backup created.")
print("Total videos:", len(df))
print(df["split"].value_counts())

display(FileLink(manifest_path))

In [ ]:

import time
import torch
from tqdm.auto import tqdm

def train_one_epoch(loader):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    start = time.time()

    progress = tqdm(loader, desc="Training Epoch 1")

    for videos, labels in progress:

        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Uniform selection: 4 frames from 29 candidates
        videos = uniform_sampling(videos, budget=4)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{100 * correct / total:.2f}%"
        )

    return {
        "train_loss": total_loss / total,
        "train_accuracy": 100 * correct / total,
        "training_seconds": time.time() - start
    }


# Train epoch 1
results = train_one_epoch(train_loader)

print("\nTRAINING COMPLETED")
print("Training loss:", round(results["train_loss"], 4))
print("Training accuracy:", round(results["train_accuracy"], 2), "%")
print("Training time:", round(results["training_seconds"], 2), "seconds")

# Save checkpoint immediately
save_checkpoint(
    model,
    optimizer,
    epoch=1,
    filename="mobilenetv3_uniform_4frames_epoch1.pth"
)

print("\nFIRST BASELINE CHECKPOINT SAVED.")



In [ ]:

import os
import zipfile
import hashlib

checkpoint_path = (
    "/kaggle/working/HAR_Research/checkpoints/"
    "mobilenetv3_uniform_4frames_epoch1.pth"
)

assert os.path.exists(checkpoint_path), "Checkpoint missing!"

size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)

# Calculate SHA256 checksum for backup verification
sha256 = hashlib.sha256()

with open(checkpoint_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

print("CHECKPOINT VERIFIED")
print("File:", os.path.basename(checkpoint_path))
print("Size:", round(size_mb, 2), "MB")
print("SHA256:", sha256.hexdigest())

# Create a ZIP backup
zip_path = (
    "/kaggle/working/"
    "mobilenetv3_uniform_4frames_epoch1_backup.zip"
)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
    z.write(
        checkpoint_path,
        arcname=os.path.basename(checkpoint_path)
    )

print("\nZIP BACKUP CREATED")
print("Path:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")


In [37]:

import torch
from tqdm.auto import tqdm

# Evaluate validation set
model.eval()

total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():

    for videos, labels in tqdm(val_loader, desc="Validation"):

        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Same 4-frame uniform sampling protocol
        videos = uniform_sampling(videos, budget=4)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

val_loss = total_loss / total
val_accuracy = 100 * correct / total

print("\nVALIDATION RESULTS")
print("------------------")
print("Validation videos:", total)
print("Validation loss:", round(val_loss, 4))
print("Validation accuracy:", round(val_accuracy, 2), "%")


Validation:   0%|          | 0/802 [00:00<?, ?it/s]


VALIDATION RESULTS
------------------
Validation videos: 1604
Validation loss: 1.8577
Validation accuracy: 53.87 %


In [39]:

# Train the second epoch
results_epoch2 = train_one_epoch(train_loader)

print("\nEPOCH 2 TRAINING RESULTS")
print("Training loss:", round(results_epoch2["train_loss"], 4))
print("Training accuracy:", round(results_epoch2["train_accuracy"], 2), "%")
print("Training time:", round(results_epoch2["training_seconds"], 2), "seconds")

# Save checkpoint immediately
save_checkpoint(
    model,
    optimizer,
    epoch=2,
    filename="mobilenetv3_uniform_4frames_epoch2.pth"
)

print("\nEPOCH 2 CHECKPOINT SAVED")



Training Epoch 1:   0%|          | 0/3967 [00:00<?, ?it/s]


EPOCH 2 TRAINING RESULTS
Training loss: 0.9614
Training accuracy: 73.68 %
Training time: 948.29 seconds
Checkpoint saved: /kaggle/working/HAR_Research/checkpoints/mobilenetv3_uniform_4frames_epoch2.pth
Size (MB): 49.92

EPOCH 2 CHECKPOINT SAVED


In [41]:
model.eval()

total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for videos, labels in tqdm(val_loader, desc="Validation Epoch 2"):
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        videos = uniform_sampling(videos, budget=4)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

val_loss = total_loss / total
val_accuracy = 100 * correct / total

print("\nEPOCH 2 VALIDATION RESULTS")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.2f}%")
print(f"Validation samples: {total}")


Validation Epoch 2:   0%|          | 0/802 [00:00<?, ?it/s]


EPOCH 2 VALIDATION RESULTS
Validation loss: 1.8382
Validation accuracy: 63.47%
Validation samples: 1604


In [42]:
import os
import zipfile
import hashlib

results_path = "/kaggle/working/HAR_Research/results/baseline_experiment_results.csv"

assert os.path.exists(results_path), "Results CSV not found!"

zip_path = "/kaggle/working/baseline_experiment_results_backup.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(results_path, arcname="baseline_experiment_results.csv")

sha256 = hashlib.sha256(open(results_path, "rb").read()).hexdigest()

print("Backup created:", zip_path)
print("CSV size:", round(os.path.getsize(results_path) / 1024, 2), "KB")
print("SHA256:", sha256)

                 experiment  epoch  candidate_frames  selected_frames  train_samples  validation_samples  train_loss  train_accuracy  validation_loss  validation_accuracy  training_time_seconds
MobileNetV3_Uniform_4Frames      1                29                4           7933                1604      2.8082           33.48         1.857700            53.870000                 936.15
MobileNetV3_Uniform_4Frames      2                29                4           7933                1604      0.9614           73.68         1.838208            63.466334                 948.29

Results saved: /kaggle/working/HAR_Research/results/baseline_experiment_results.csv


In [46]:
# ==========================================
# EPOCH 3 - BASELINE TRAINING
# MobileNetV3-Large | Uniform Sampling
# ==========================================

import time
import torch
from tqdm.auto import tqdm

model.train()

start_time = time.time()

total_loss = 0.0
correct = 0
total = 0

progress_bar = tqdm(train_loader, desc="Training Epoch 3")

for videos, labels in progress_bar:

    videos = videos.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)

    # Uniformly select 4 frames from 29 candidates
    videos = uniform_sampling(videos, budget=4)

    optimizer.zero_grad(set_to_none=True)

    outputs = model(videos)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    batch_size = labels.size(0)

    total_loss += loss.item() * batch_size
    correct += (outputs.argmax(1) == labels).sum().item()
    total += batch_size

    progress_bar.set_postfix({
        "loss": f"{total_loss / total:.4f}",
        "accuracy": f"{100 * correct / total:.2f}%"
    })

training_time = time.time() - start_time

train_loss = total_loss / total
train_accuracy = 100 * correct / total

print("\nEPOCH 3 TRAINING RESULTS")
print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.2f}%")
print(f"Training time: {training_time:.2f} seconds")

# Save Epoch 3 checkpoint
save_checkpoint(
    model,
    optimizer,
    epoch=3,
    filename="mobilenetv3_uniform_4frames_epoch3.pth"
)

Training Epoch 3:   0%|          | 0/3967 [00:00<?, ?it/s]


EPOCH 3 TRAINING RESULTS
Training loss: 0.4613
Training accuracy: 87.10%
Training time: 946.49 seconds
CHECKPOINT SAVED SUCCESSFULLY
File: /kaggle/working/HAR_Research/checkpoints/mobilenetv3_uniform_4frames_epoch3.pth
Epoch: 3
Backbone: MobileNetV3-Large
Dataset: UCF101
Candidate frames: 29
Selected frames: 4
Sampling: Uniform
File size: 49.92 MB


'/kaggle/working/HAR_Research/checkpoints/mobilenetv3_uniform_4frames_epoch3.pth'

In [47]:
import os
import zipfile
import hashlib

checkpoint = (
    "/kaggle/working/HAR_Research/checkpoints/"
    "mobilenetv3_uniform_4frames_epoch3.pth"
)

assert os.path.exists(checkpoint), "Checkpoint not found!"

zip_path = (
    "/kaggle/working/"
    "mobilenetv3_uniform_4frames_epoch3_backup.zip"
)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
    z.write(checkpoint, arcname=os.path.basename(checkpoint))

sha256 = hashlib.sha256(open(checkpoint, "rb").read()).hexdigest()

print("Checkpoint:", os.path.basename(checkpoint))
print("Size:", round(os.path.getsize(checkpoint) / 1024**2, 2), "MB")
print("SHA256:", sha256)
print("Backup ZIP:", zip_path)
print("ZIP size:", round(os.path.getsize(zip_path) / 1024**2, 2), "MB")

Checkpoint: mobilenetv3_uniform_4frames_epoch3.pth
Size: 49.92 MB
SHA256: ad6a16a73a68700081f2feae5cfc3505bd1ab3310255777302c57e0b03cedb64
Backup ZIP: /kaggle/working/mobilenetv3_uniform_4frames_epoch3_backup.zip
ZIP size: 49.92 MB


## Dataset Partitioning and Leakage Prevention

The UCF101 dataset was partitioned using the official Split 1 training and testing lists.

The official training data was further divided into training and validation subsets using a group-based stratified procedure within each action class.

- **Training:** 7,933 videos
- **Validation:** 1,604 videos
- **Testing:** 3,783 videos
- **Total:** 13,320 videos
- **Action classes:** 101

A fixed random seed (`random_state=42`) was used to ensure reproducibility. Source-group overlap between the training and validation subsets was checked and confirmed to be absent.

The official test set remains separate and will be reserved for final evaluation.

**Leakage Check:** PASSED.

### Stage 2: Dataset Integrity Audit — Completed

The complete UCF101 video integrity audit was successfully repeated.

- Total videos audited: 13,320
- Fully readable videos: 13,096
- Partially readable videos: 224
- Unreadable videos: 0
- Audit duration: 772.52 seconds

All videos contain at least one successfully decoded frame. Partially readable videos are retained using their actual decoded frame counts.

**Output:** `full_video_integrity_audit.csv`

**Next:** Restore the audited manifest, video Dataset class, and DataLoaders.

### Project Recovery — November 22, 2026

The Kaggle session restarted and the previous training checkpoint was not preserved.

**Previously completed work:**
- UCF101 dataset preparation and integrity audit.
- 32-frame candidate extraction pipeline.
- Uniform and temporal-difference sampling methods.
- MobileNetV3-Large video classifier.
- First baseline training epoch.

**Previously recorded baseline results:**
- Training videos: 7,933
- Frame budget: 4
- Training loss: 2.7532
- Training accuracy: 34.35%
- Training time: 901.24 seconds

**Recovery plan:** Restore the dataset pipeline, retrain the baseline, and immediately download the model checkpoint to local storage.

### Stage 1: Dataset Preparation and Model Pipeline Validation

**Completed tasks:**
- Verified the UCF101 dataset containing 13,320 videos across 101 action classes.
- Prepared training (7,933), validation (1,604), and testing (3,783) partitions.
- Completed video integrity auditing and generated an audited dataset manifest.
- Developed and tested sequential video decoding and 32-frame candidate extraction.
- Implemented and validated two frame-selection strategies:
  - Uniform temporal sampling
  - Temporal-difference-based adaptive sampling
- Tested frame budgets of 4, 8, 12, and 16.
- Configured MobileNetV3-Large with a 101-class output layer.
- Successfully completed a 10-batch GPU training pilot.

**Initial status:** The data pipeline, sampling methods, classification model, and initial GPU training workflow are operational.

**Next stage:** Conduct controlled baseline and proposed-method experiments, followed by validation and computational-efficiency evaluation.

*Note: The pilot is a pipeline test, not a final model-performance evaluation.*